# 01 — Download MPI-LEMON Phenotypic Data

**Run on Kaggle:** New Notebook → Settings: GPU on, Internet on

Downloads `participants.tsv`, discovers the correct S3 file naming pattern,
and builds the subject list for notebook 02.

**After running:** Save Version → tick **"Save output files as dataset"** → name it `lemon-meta`.

In [ ]:
!pip install -q awscli nibabel

In [ ]:
import os, json, subprocess
import pandas as pd
import numpy as np

BUCKET = 's3://openneuro.org/ds000221'
print('Bucket:', BUCKET)

In [ ]:
# ── Download participants.tsv ──────────────────────────────────────────────────
!aws s3 cp {BUCKET}/participants.tsv /kaggle/working/participants.tsv --no-sign-request

participants = pd.read_csv('/kaggle/working/participants.tsv', sep='\t')
print(f'Participants: {len(participants)}')
print(f'Columns: {participants.columns.tolist()}')
participants.head()

In [ ]:
# ── List subjects on S3 ───────────────────────────────────────────────────────
result = subprocess.run(
    ['aws', 's3', 'ls', f'{BUCKET}/', '--no-sign-request'],
    capture_output=True, text=True
)
subjects = sorted([
    l.strip().rstrip('/').split()[-1]
    for l in result.stdout.strip().split('\n') if 'sub-' in l
])
print(f'Subjects on S3: {len(subjects)}')
print('First 5:', subjects[:5])

In [ ]:
# ── Discover the actual file naming pattern for this dataset ──────────────────
# OpenNeuro datasets sometimes use non-standard task names or additional BIDS
# labels (run, acq, etc.). We list one subject's func/ directory to find out.

test_sub = subjects[0]
print(f'Inspecting {test_sub}/func/ on S3...')

ls_result = subprocess.run(
    ['aws', 's3', 'ls', f'{BUCKET}/{test_sub}/func/', '--no-sign-request'],
    capture_output=True, text=True
)

if ls_result.returncode != 0 or not ls_result.stdout.strip():
    print('No func/ directory found for', test_sub)
    print('Trying without func/ subdirectory...')
    ls_result = subprocess.run(
        ['aws', 's3', 'ls', f'{BUCKET}/{test_sub}/', '--no-sign-request'],
        capture_output=True, text=True
    )

print('Files found:')
print(ls_result.stdout)

# Extract all .nii.gz filenames
func_files = [
    l.strip().split()[-1]
    for l in ls_result.stdout.strip().split('\n')
    if '.nii.gz' in l or '.nii' in l
]
print('NIfTI files:', func_files)

In [ ]:
# ── Identify the resting-state BOLD file ─────────────────────────────────────
# Look for files containing 'rest' and 'bold' in their name.

bold_candidates = [f for f in func_files if 'bold' in f.lower() or 'rest' in f.lower()]
print('BOLD candidates:', bold_candidates)

if not bold_candidates:
    # Fallback: list ALL files recursively for this subject to find what exists
    print('\nNo obvious BOLD file found. Listing full subject tree...')
    full_ls = subprocess.run(
        ['aws', 's3', 'ls', f'{BUCKET}/{test_sub}/', '--recursive', '--no-sign-request'],
        capture_output=True, text=True
    )
    print(full_ls.stdout[:3000])  # first 3000 chars
    raise RuntimeError('Could not identify BOLD file. See listing above and update BOLD_SUFFIX below.')

# The filename without the subject prefix is the suffix pattern
# e.g. 'sub-010002_task-rest_bold.nii.gz' → suffix = '_task-rest_bold.nii.gz'
bold_file   = bold_candidates[0]
BOLD_SUFFIX = bold_file.replace(test_sub, '')   # remove subject prefix
FUNC_SUBDIR = 'func/' if 'func/' in ls_result.args[-1] else ''

print(f'\nBOLD suffix pattern: "{BOLD_SUFFIX}"')
print(f'Func subdir: "{FUNC_SUBDIR}"')

# Save the pattern so notebook 02 uses it
path_config = {'bold_suffix': BOLD_SUFFIX, 'func_subdir': FUNC_SUBDIR}
with open('/kaggle/working/path_config.json', 'w') as f:
    json.dump(path_config, f, indent=2)
print('path_config.json saved ✓')

In [ ]:
# ── Test: download the identified file for one subject ────────────────────────
import nibabel as nib

bold_key  = f'{BUCKET}/{test_sub}/{FUNC_SUBDIR}{test_sub}{BOLD_SUFFIX}'
test_path = f'/kaggle/working/{test_sub}_test.nii.gz'

print(f'Trying: {bold_key}')
r = subprocess.run(
    ['aws', 's3', 'cp', bold_key, test_path, '--no-sign-request'],
    capture_output=True, text=True
)
if r.returncode != 0:
    print('ERROR:', r.stderr)
    raise RuntimeError(f'Download failed. Check the S3 listing above for the correct path.')

img = nib.load(test_path)
print(f'Shape: {img.shape}   TR: {img.header.get_zooms()[3]:.3f}s')
os.remove(test_path)
print('S3 access confirmed ✓')

In [ ]:
# ── Find phenotypic target columns ────────────────────────────────────────────
TARGET_PATTERNS = {
    'trait_anxiety':  ['STAI_T', 'STAI_Trait', 'stai_t', 'trait_anxiety'],
    'chronic_stress': ['TICS_SSCS', 'TICS', 'tics_sscs', 'chronic_stress'],
    'neuroticism':    ['NEO_N', 'neo_n', 'neuroticism', 'NEO_FFI_N'],
}

found = {}
print('Scanning columns for targets...')
for name, patterns in TARGET_PATTERNS.items():
    for p in patterns:
        matches = [c for c in participants.columns if p.lower() in c.lower()]
        if matches:
            found[name] = matches[0]
            vals = participants[matches[0]].dropna().values
            print(f'  {name:20s} → "{matches[0]}"  (n={len(vals)}, mean={vals.mean():.1f})')
            break
    if name not in found:
        print(f'  {name:20s} → NOT FOUND in columns')

if not found:
    print('\nNo target columns found. All columns:')
    print(participants.columns.tolist())
    print('\nUpdate TARGET_PATTERNS above to match the column names shown.')

with open('/kaggle/working/target_columns.json', 'w') as f:
    json.dump(found, f, indent=2)
print('target_columns.json saved ✓')

In [ ]:
# ── Build final subject list ───────────────────────────────────────────────────
# If no phenotypic targets were found, still save all subjects so preprocessing
# can run — targets can be matched later from a separate phenotypic file.

if found:
    target_cols = list(found.values())
    valid       = participants.dropna(subset=target_cols).copy()
    valid_ids   = set(valid['participant_id'].str.replace('sub-', '', regex=False))
    final_subjects = [s for s in subjects if s.replace('sub-', '') in valid_ids]
    valid.to_csv('/kaggle/working/phenotypic.csv', index=False)
    print(f'Subjects with complete phenotypic data: {len(final_subjects)}')
else:
    # No matched columns — save all subjects and full participants file
    final_subjects = subjects
    participants.to_csv('/kaggle/working/phenotypic.csv', index=False)
    print(f'WARNING: targets not matched — saving all {len(subjects)} subjects.')
    print('Update TARGET_PATTERNS in the cell above then re-run.')

with open('/kaggle/working/subject_list.json', 'w') as f:
    json.dump(final_subjects, f, indent=2)
print('subject_list.json saved ✓')

In [ ]:
print('='*55)
print('NOTEBOOK 01 COMPLETE')
print('='*55)
print(f'  Subjects ready:  {len(final_subjects)}')
print(f'  BOLD suffix:     {BOLD_SUFFIX}')
print(f'  Targets found:   {list(found.keys())}')
print()
print('FILES IN /kaggle/working:')
for fname in sorted(os.listdir('/kaggle/working')):
    size = os.path.getsize(f'/kaggle/working/{fname}')
    print(f'  {fname}  ({size/1024:.1f} KB)')
print()
print('NEXT STEPS:')
print('  1. Save Version → tick "Save output files as dataset" → name: lemon-meta')
print('  2. Open notebook 02, attach lemon-meta, run preprocessing')